# Quantitative Analysis (SIM/SRM)

This notebook demonstrates a practical workflow for quantitative (**SIM/SRM**) MS data in an interactive Python environment.
**QuanAnalysis** is a Hive data object that represents a mass chromatogram acquired by the Selected Ion Monitoring (SIM) or Selected Reaction Monitoring (SRM) mode.

You will use:

- `hive_data_access` to open a Hive file and access data objects
- `hivetoolkit` to visualize chromatograms


## Preparation

### Importing required libraries

This notebook assumes that `hive_data_access` and `hivetoolkit` are available in the current Python environment.  
When you install `hivetoolkit` using `pip install ./hivetoolkit`, the `hive-data-access` package will also be installed from PyPI.

Note: Reading `.hmD` data may require a Hive Runtime subscription or license, whereas `.mzB` files are typically readable without license verification.

In [ ]:
import hive_data_access
import hivetoolkit

## File Access

Open a sample file as a `HiveFile` object. This is the entry point for accessing measurements and analyses via `hive_data_access`.


In [ ]:
from pathlib import Path

# set the file paths to the Hive format files
quan_file_path1 = Path('../sample/sample_quan1.mzB')
quan_file_path2 = Path('../sample/sample_quan2.mzB')

# open hive files
hive1 = hive_data_access.HiveFile(quan_file_path1)
hive2 = hive_data_access.HiveFile(quan_file_path2)

## Measurement

A `Measurement` represents one sample within a file. Most files contain a single measurement, but some vendor formats can contain multiple.

- Check how many measurements exist: `hive.get_measurement_count()`
- Retrieve one measurement: `hive.get_measurement(i)` (i: zero-based index)

A `Measurement` also provides metadata such as sample name and acquisition time.


In [ ]:
# check the number of measurements
print('Measurement Count: ' + str(hive1.get_measurement_count()))

# select first measurement
measurement1 = hive1.get_measurement(0)

# show part of the measurement information as an example
print('Sample Name: ' + str(measurement1.sample_name))
print('Sample ID: ' + str(measurement1.sample_id))
print('Measurement Datetime: ' + str(measurement1.measurement_date_time))
print('Injection Volume: ' + str(measurement1.injection_volume))
print('Dilution Factor: ' + str(measurement1.dilution_factor))
print('Plate Code: ' + str(measurement1.plate_code))
print('Well Code: ' + str(measurement1.well_code))
print('Instrument Manufacturer: ' + str(measurement1.instrument_manufacturer))
print('Instrument Name: ' + str(measurement1.instrument_product_name))

## QuanAnalysis

SIM/SRM data are provided as a collection of `QuanAnalysis` objects under a `Measurement`.

- Confirm how many analyses exist: `measurement.get_quan_analysis_count()`
- Retrieve one analysis: `measurement.get_quan_analysis(i)` (i: zero-based index)

Each `QuanAnalysis` typically corresponds to one SIM or SRM channel. It also provides metadata such as Q1/Q3 (SRM) or extract value (SIM).


In [ ]:
# check the number of QuanAnalysis
print('QuanAnalysis Count: ' + str(measurement1.get_quan_analysis_count()))

# select first QuanAnalysis
quan1 = measurement1.get_quan_analysis(0)

# show a part of the QuanAnalysis information as an example
print('Compound Name: ' + str(quan1.compound_name))
print('Compound Type: ' + str(quan1.compound_type))
print('Concentration: ' + str(quan1.concentration))
print('Quan Analysis Type: ' + str(quan1.quan_analysis_type))
print('Polarity: ' + str(quan1.polarity))
print('Q1 > Q3: ' + str(quan1.q1) + ' > ' + str(quan1.q3))
print('Collision Energy: ' + str(quan1.collision_energy))
print('Estimated Retention Time: ' + str(quan1.estimated_retention_time))

### Single Chromatogram

To plot one chromatogram, pass a `QuanAnalysis` object directly:

- `hivetoolkit.draw_chromatogram(quan_analysis)`

You can also pass a list of points (e.g., `quan_analysis.chromatogram_points`), but providing the `QuanAnalysis` object is convenient because labels can be inferred automatically.


In [ ]:
# get a chromatogram of the single QuanAnalysis
points = quan1.chromatogram_points # not necessary when directly specifying the QuanAnalysis in draw_chromatogram (below)

# draw a chromatogram (labels arguments are optional)
# In this example, 3 labels are defined:
#  First label is drawn at the top peak, with the specified format.
#  2nd and 3rd labels are fixed labels, at the fixed RT values (8.50 and 9.05) with the fixed label strings.
hivetoolkit.draw_chromatogram(quan1, labels={'top': 'RT={x:.3f}, Intensity={y:.1e}', 'offset': 25, 8.50: 'RT=8.50', 9.05: 'RT=9.05'})
# color='darkblue', style=dict(font='sans', spac='comp'), ## other possible optional arguments;

# hivetoolkit.draw_chromatogram(points, title = quan1.compound_name) # another way to draw the same chromatogram, using points argument

### Multiple Chromatograms

To overlay multiple chromatograms, pass a list of `QuanAnalysis` objects:

- `hivetoolkit.draw_overlaid_chromatogram([qa1, qa2, qa3], colors=...)`

If you prefer, you can pass a list of point lists instead. When providing colors, use any valid Python/Plotly color specification.


In [ ]:
# read the first measurement from the other file
measurement2 = hive2.get_measurement(0)

# search the same compound and the same transitions in different samples
for i in range(measurement2.get_quan_analysis_count()):
    quan2 = measurement2.get_quan_analysis(i)
    if quan1.compound_name == quan2.compound_name and quan1.q1 == quan2.q1 and quan1.q3 == quan2.q3:
        break

# set the plot data for each sample
points1 = quan1.chromatogram_points # not necessary when directly specifying the QuanAnalysis in draw_overlaid_chromatogram (below)
points2 = quan2.chromatogram_points # not necessary when directly specifying the QuanAnalysis in draw_overlaid_chromatogram

# draw overlaid chromatograms
# In this example, chromatogram colors are defined by a list, specifying the first and second chromatogram colors respectively.
hivetoolkit.draw_overlaid_chromatogram([quan1, quan2], title = 'Sample-1 & Sample-2', colors=['darkred', 'orange'])
# style = dict(font='serif', spac='spacious'), ## other possible optional arguments

# hivetoolkit.draw_overlaid_chromatogram([points1, points2], title = 'Sample-1 & Sample-2') # another way to draw the same chromatograms, using a list of points argument

## Termination

When you are done, close the file handle:

- `hive.close()`


In [ ]:
# close opened files
hive1.close()
hive2.close()